In [ ]:
# Save synthetic dataset
print(f"\n✓ Saving synthetic dataset...")
data_path = '../data/synthetic_attendance_data.csv'
df.to_csv(data_path, index=False)
print(f"  Dataset saved: {data_path}")
print(f"  Records: {len(df)}")

print("\n" + "="*60)
print("✓ PHASE 1 COMPLETED SUCCESSFULLY!")
print("="*60)
print(f"\nSummary:")
print(f"  ✓ Synthetic dataset generated: 1000 records")
print(f"  ✓ Random Forest model trained with {model.n_estimators} trees")
print(f"  ✓ Model accuracy: {test_accuracy*100:.2f}% (Target: ≥85%)")
print(f"  ✓ Model saved and ready for backend integration")
print(f"  ✓ All visualizations and reports generated")
print(f"\nNext Step: PHASE 2 - Backend Integration (ML Prediction Endpoints)")

In [ ]:
# Save Model and Supporting Files
print("\n" + "="*60)
print("SAVING MODEL")
print("="*60)

# Save model
model_path = '../models/random_forest_model.pkl'
joblib.dump(model, model_path)
print(f"\n✓ Model saved: {model_path}")
print(f"  File size: {pd.Series([model_path]).apply(lambda x: f'{x}').values[0]}")

# Save feature names for later use
features_info = {
    'feature_names': feature_columns,
    'classes': model.classes_.tolist(),
    'training_accuracy': float(train_accuracy),
    'testing_accuracy': float(test_accuracy)
}

print(f"\n✓ Model Information:")
print(f"  Input features: {len(feature_columns)}")
print(f"  Output classes: {len(model.classes_)}")
print(f"  Classes: {', '.join(model.classes_)}")
print(f"  Training accuracy: {train_accuracy*100:.2f}%")
print(f"  Testing accuracy: {test_accuracy*100:.2f}%")

## 7. Save Trained Model

Serialize and save the model for backend integration.

In [ ]:
# Feature Importance Analysis
print(f"\n5. Feature Importance Analysis:")
print("="*60)

feature_importance = pd.DataFrame({
    'feature': feature_columns,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

print(feature_importance.to_string(index=False))

# Visualize Feature Importance
fig, ax = plt.subplots(figsize=(10, 6))
feature_importance.plot(x='feature', y='importance', kind='barh', ax=ax, color='steelblue', legend=False)
ax.set_title('Feature Importance in Random Forest Model', fontsize=12, fontweight='bold')
ax.set_xlabel('Importance Score')
ax.invert_yaxis()
plt.tight_layout()
plt.show()

print(f"\nTop 3 Most Important Features:")
for idx, row in feature_importance.head(3).iterrows():
    print(f"  {idx+1}. {row['feature']}: {row['importance']:.4f}")

In [ ]:
# Confusion Matrix
print(f"\n4. Confusion Matrix (Test Set):")
print("="*60)
cm = confusion_matrix(y_test, y_pred_test)
print(cm)

# Visualize Confusion Matrix
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Confusion Matrix Heatmap
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0], 
            xticklabels=['Regular', 'Irregular', 'High_Risk'],
            yticklabels=['Regular', 'Irregular', 'High_Risk'],
            cbar_kws={'label': 'Count'})
axes[0].set_title('Confusion Matrix (Test Set)', fontsize=12, fontweight='bold')
axes[0].set_ylabel('True Label')
axes[0].set_xlabel('Predicted Label')

# Accuracy Comparison
accuracies = [train_accuracy, test_accuracy]
colors = ['#2ecc71' if acc >= 0.85 else '#f39c12' for acc in accuracies]
axes[1].bar(['Training', 'Testing'], [acc*100 for acc in accuracies], color=colors)
axes[1].axhline(y=85, color='red', linestyle='--', linewidth=2, label='Target (85%)')
axes[1].set_title('Model Accuracy', fontsize=12, fontweight='bold')
axes[1].set_ylabel('Accuracy (%)')
axes[1].set_ylim([0, 105])
for i, (label, acc) in enumerate(zip(['Training', 'Testing'], accuracies)):
    axes[1].text(i, acc*100 + 2, f'{acc*100:.2f}%', ha='center', fontweight='bold')
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
# Classification Report
print(f"\n3. Classification Report (Test Set):")
print("="*60)
print(classification_report(y_test, y_pred_test))

In [ ]:
# Model Predictions
print("\n" + "="*60)
print("MODEL EVALUATION")
print("="*60)

print("\n1. Making predictions...")
y_pred_train = model.predict(X_train)
y_pred_test = model.predict(X_test)

print("   ✓ Predictions completed")

# Calculate Accuracy
train_accuracy = accuracy_score(y_train, y_pred_train)
test_accuracy = accuracy_score(y_test, y_pred_test)

print(f"\n2. Accuracy Metrics:")
print(f"   Training Accuracy: {train_accuracy:.4f} ({train_accuracy*100:.2f}%)")
print(f"   Testing Accuracy: {test_accuracy:.4f} ({test_accuracy*100:.2f}%)")

if test_accuracy >= 0.85:
    print(f"\n   ✓ MODEL MEETS TARGET ACCURACY (≥85%)!")
else:
    print(f"\n   ⚠ Model accuracy is {test_accuracy*100:.2f}%, slightly below 85% target")

## 6. Model Evaluation & Validation

Evaluate model performance using accuracy, precision, recall, F1-score, and confusion matrix.

In [ ]:
# Train Random Forest Model
print("\n2. Training Random Forest Classifier...")

model = RandomForestClassifier(
    n_estimators=100,           # Number of trees
    max_depth=15,               # Maximum tree depth
    min_samples_split=5,        # Minimum samples to split
    min_samples_leaf=2,         # Minimum samples per leaf
    random_state=42,
    n_jobs=-1,                  # Use all processors
    class_weight='balanced'     # Balance classes
)

model.fit(X_train, y_train)
print("   ✓ Model training completed!")

# Model hyperparameters
print(f"\nModel Hyperparameters:")
print(f"  - Number of trees: {model.n_estimators}")
print(f"  - Max depth: {model.max_depth}")
print(f"  - Min samples split: {model.min_samples_split}")
print(f"  - Min samples leaf: {model.min_samples_leaf}")
print(f"  - Class weight: balanced")

In [ ]:
# Train-Test Split
print("="*60)
print("MODEL TRAINING")
print("="*60)

print("\n1. Splitting data into training and testing sets...")
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"   ✓ Data split completed")
print(f"\nTraining set size: {X_train.shape[0]} ({X_train.shape[0]/len(X)*100:.1f}%)")
print(f"Testing set size: {X_test.shape[0]} ({X_test.shape[0]/len(X)*100:.1f}%)")

print(f"\nTraining set class distribution:")
print(y_train.value_counts())
print(f"\nTesting set class distribution:")
print(y_test.value_counts())

## 5. Train Random Forest Classifier

Split data and train the Random Forest model with optimized hyperparameters.

In [ ]:
# Create Feature Matrix
print("\n2. Creating feature matrix...")

feature_columns = [
    'is_late',
    'absences_30d',
    'attendance_rate',
    'overtime_hours',
    'total_hours_worked',
    'day_of_week',
    'check_in_hour',
    'check_out_hour',
    'shift_type_encoded',
    'is_present'
]

# Prepare X and y
X = df_processed[feature_columns].fillna(0)
y = df_processed['worker_category']

print(f"   ✓ Feature matrix created")
print(f"\nFeature Matrix Shape: {X.shape}")
print(f"Target Variable Shape: {y.shape}")
print(f"\nFeatures used:")
for i, col in enumerate(feature_columns, 1):
    print(f"  {i}. {col}")

# Display feature statistics
print(f"\nFeature Statistics:")
print(X.describe())

In [ ]:
# Data Preprocessing
print("="*60)
print("DATA PREPROCESSING")
print("="*60)

df_processed = df.copy()

# Convert date to datetime
df_processed['date'] = pd.to_datetime(df_processed['date'])

# Feature Engineering
print("\n1. Creating derived features...")

# Day of week
df_processed['day_of_week'] = df_processed['date'].dt.dayofweek

# Extract hour from check_in
df_processed['check_in_hour'] = df_processed['check_in'].fillna('00:00').apply(lambda x: int(x.split(':')[0]))
df_processed['check_out_hour'] = df_processed['check_out'].fillna('00:00').apply(lambda x: int(x.split(':')[0]))

# Encode shift type
shift_encoder = LabelEncoder()
df_processed['shift_type_encoded'] = shift_encoder.fit_transform(df_processed['shift_type'].fillna('Unknown'))

# Presence indicator
df_processed['is_present'] = (~df_processed['check_in'].isna()).astype(int)

# Presence rate by worker
df_processed['presence_rate_worker'] = df_processed.groupby('worker_id')['is_present'].transform('mean')

print("   ✓ Derived features created")
print(f"\nFeatures created: day_of_week, check_in_hour, check_out_hour, shift_type_encoded, is_present, presence_rate_worker")

## 4. Data Preprocessing & Feature Engineering

Prepare data for model training by handling missing values, encoding categorical features, and creating derived features.

In [ ]:
# Feature Distribution Analysis
print("\n" + "="*60)
print("FEATURE DISTRIBUTIONS BY WORKER CATEGORY")
print("="*60)

fig, axes = plt.subplots(2, 3, figsize=(16, 10))

# Attendance Rate
df.boxplot(column='attendance_rate', by='worker_category', ax=axes[0, 0])
axes[0, 0].set_title('Attendance Rate Distribution')
axes[0, 0].set_xlabel('Worker Category')
axes[0, 0].set_ylabel('Attendance Rate')

# Absences 30d
df.boxplot(column='absences_30d', by='worker_category', ax=axes[0, 1])
axes[0, 1].set_title('Absences in Last 30 Days')
axes[0, 1].set_xlabel('Worker Category')
axes[0, 1].set_ylabel('Count')

# Overtime Hours
df.boxplot(column='overtime_hours', by='worker_category', ax=axes[0, 2])
axes[0, 2].set_title('Overtime Hours Distribution')
axes[0, 2].set_xlabel('Worker Category')
axes[0, 2].set_ylabel('Hours')

# Late Arrivals
late_by_category = df.groupby('worker_category')['is_late'].sum()
late_by_category.plot(kind='bar', ax=axes[1, 0], color=['#2ecc71', '#f39c12', '#e74c3c'])
axes[1, 0].set_title('Late Arrivals by Category')
axes[1, 0].set_ylabel('Count')
axes[1, 0].set_xlabel('Worker Category')
plt.setp(axes[1, 0].xaxis.get_majorticklabels(), rotation=45)

# Total Hours Worked
df.boxplot(column='total_hours_worked', by='worker_category', ax=axes[1, 1])
axes[1, 1].set_title('Total Hours Worked Distribution')
axes[1, 1].set_xlabel('Worker Category')
axes[1, 1].set_ylabel('Hours')

# Shift Type Distribution
shift_dist = pd.crosstab(df['shift_type'], df['worker_category'], normalize='columns') * 100
shift_dist.plot(kind='bar', ax=axes[1, 2], color=['#2ecc71', '#f39c12', '#e74c3c'])
axes[1, 2].set_title('Shift Type by Category (%)')
axes[1, 2].set_ylabel('Percentage')
axes[1, 2].set_xlabel('Shift Type')
plt.setp(axes[1, 2].xaxis.get_majorticklabels(), rotation=45)

plt.suptitle('Feature Distribution Analysis by Worker Category', fontsize=14, fontweight='bold', y=1.00)
plt.tight_layout()
plt.show()

In [ ]:
# Class Distribution Analysis
print("\n" + "="*60)
print("CLASS DISTRIBUTION")
print("="*60)
class_dist = df['worker_category'].value_counts()
print(class_dist)
print(f"\nClass Distribution (%):")
print(df['worker_category'].value_counts(normalize=True) * 100)

# Visualize class distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Pie chart
df['worker_category'].value_counts().plot(kind='pie', ax=axes[0], autopct='%1.1f%%', colors=['#2ecc71', '#f39c12', '#e74c3c'])
axes[0].set_title('Worker Category Distribution', fontsize=12, fontweight='bold')
axes[0].set_ylabel('')

# Bar chart
df['worker_category'].value_counts().plot(kind='bar', ax=axes[1], color=['#2ecc71', '#f39c12', '#e74c3c'])
axes[1].set_title('Worker Category Count', fontsize=12, fontweight='bold')
axes[1].set_ylabel('Count')
axes[1].set_xlabel('')
plt.setp(axes[1].xaxis.get_majorticklabels(), rotation=45)

plt.tight_layout()
plt.show()

In [ ]:
# Dataset Overview
print("="*60)
print("DATASET OVERVIEW")
print("="*60)
print(f"\nDataset Shape: {df.shape}")
print(f"Total Records: {len(df)}")
print(f"Total Workers: {df['worker_id'].nunique()}")
print(f"Total Contractors: {df['contractor_id'].nunique()}")
print(f"\nData Types:")
print(df.dtypes)
print(f"\nMissing Values:")
print(df.isnull().sum())
print(f"\nBasic Statistics:")
print(df.describe())

## 3. Exploratory Data Analysis (EDA)

Analyze data distributions, patterns, and relationships.

In [ ]:
def generate_synthetic_dataset(n_samples=1000):
    """Generate synthetic attendance dataset with realistic patterns"""
    
    n_workers = 150
    n_contractors = 5
    
    data = {
        'worker_id': [], 'name': [], 'contractor_id': [], 'date': [],
        'check_in': [], 'check_out': [], 'shift_type': [], 'is_late': [],
        'absences_30d': [], 'attendance_rate': [], 'overtime_hours': [],
        'total_hours_worked': [], 'worker_category': []
    }
    
    base_date = datetime(2023, 1, 1)
    first_names = ['Rajesh', 'Priya', 'Amit', 'Deepak', 'Neha', 'Vikram', 'Arjun', 'Sneha', 'Rohan', 'Pooja']
    last_names = ['Kumar', 'Singh', 'Sharma', 'Patel', 'Gupta', 'Verma', 'Joshi', 'Reddy', 'Nair', 'Rao']
    
    for i in range(n_samples):
        worker_id = np.random.randint(1001, 1001 + n_workers)
        contractor_id = np.random.randint(101, 101 + n_contractors)
        
        # Determine worker category
        category = np.random.choice(['Regular', 'Irregular', 'High_Risk'], p=[0.6, 0.25, 0.15])
        
        # Generate date (skip Sundays)
        date = base_date + timedelta(days=np.random.randint(0, 365))
        while date.weekday() == 6:
            date += timedelta(days=1)
        
        # Generate attendance based on category
        attendance_prob = 0.95 if category == 'Regular' else 0.6 if category == 'Irregular' else 0.4
        
        if np.random.random() < attendance_prob:
            # Worker is present
            shift_type = np.random.choice(['Morning', 'Evening', 'Full'], p=[0.3, 0.3, 0.4])
            
            if shift_type == 'Morning':
                check_in = f"{np.random.randint(5, 8):02d}:{np.random.randint(0, 60):02d}"
                check_out = f"{np.random.randint(12, 14):02d}:{np.random.randint(0, 60):02d}"
                is_late = 1 if int(check_in.split(':')[0]) > 8 else 0
            elif shift_type == 'Evening':
                check_in = f"{np.random.randint(14, 16):02d}:{np.random.randint(0, 60):02d}"
                check_out = f"{np.random.randint(22, 24):02d}:{np.random.randint(0, 60):02d}"
                is_late = 1 if int(check_in.split(':')[0]) > 16 else 0
            else:
                check_in = f"{np.random.randint(5, 8):02d}:{np.random.randint(0, 60):02d}"
                check_out = f"{np.random.randint(17, 19):02d}:{np.random.randint(0, 60):02d}"
                is_late = 1 if int(check_in.split(':')[0]) > 8 else 0
            
            # Calculate hours worked
            ci_h = int(check_in.split(':')[0]) + int(check_in.split(':')[1])/60
            co_h = int(check_out.split(':')[0]) + int(check_out.split(':')[1])/60
            if co_h < ci_h:
                co_h += 24
            total_hours = co_h - ci_h
            overtime = np.random.choice([0, 0.5, 1, 1.5, 2], p=[0.7, 0.1, 0.1, 0.05, 0.05])
        else:
            # Worker is absent
            check_in, check_out, shift_type = None, None, None
            is_late = 0
            total_hours, overtime = 0, 0
        
        # Generate dependent features
        absences_30d = np.random.randint(0, 8 if category == 'High_Risk' else 2 if category == 'Irregular' else 0)
        attendance_rate = np.random.uniform(
            0.4, 0.6) if category == 'High_Risk' else np.random.uniform(0.75, 0.95) if category == 'Irregular' else np.random.uniform(0.95, 1.0)
        
        name = f"{random.choice(first_names)} {random.choice(last_names)}"
        
        # Append data
        data['worker_id'].append(worker_id)
        data['name'].append(name)
        data['contractor_id'].append(contractor_id)
        data['date'].append(date.strftime('%Y-%m-%d'))
        data['check_in'].append(check_in)
        data['check_out'].append(check_out)
        data['shift_type'].append(shift_type)
        data['is_late'].append(is_late)
        data['absences_30d'].append(absences_30d)
        data['attendance_rate'].append(round(attendance_rate, 2))
        data['overtime_hours'].append(round(overtime, 2))
        data['total_hours_worked'].append(round(total_hours, 2))
        data['worker_category'].append(category)
    
    return pd.DataFrame(data)

# Generate dataset
print("Generating synthetic attendance dataset...")
df = generate_synthetic_dataset(n_samples=1000)
print(f"✓ Dataset generated successfully!")
print(f"  Shape: {df.shape}")
print(f"\nFirst 10 records:")
print(df.head(10))

## 2. Generate Synthetic Attendance Dataset

Create a realistic synthetic dataset with 1000+ rows representing worker attendance patterns.

In [ ]:
# Import Required Libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay
import joblib
from datetime import datetime, timedelta
import random
import warnings
warnings.filterwarnings('ignore')

# Set random seed for reproducibility
np.random.seed(42)
random.seed(42)

# Configure visualization settings
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
print("✓ All libraries imported successfully!")

## 1. Import Required Libraries

Import all necessary libraries for data analysis, model training, and visualization.

# Worker Attendance Prediction Model
## Using Random Forest Classifier for Worker Behavior Prediction

This notebook provides a complete analysis and training pipeline for predicting worker attendance patterns:
- **Regular Worker**: Consistent attendance
- **Irregular Attendance**: Sporadic attendance patterns
- **High Absence Risk**: Frequent absenteeism

### Objective
Build a production-ready Random Forest model with ≥85% accuracy to predict worker categories based on attendance patterns.